# 🖥️ TOCAIA OSINT Linux — Desktop no Colab (ver a cara do sistema, sem instalar nada)

Esta é a versão **completa/visual**: o **desktop Xfce da TOCAIA** — wallpaper da onça,
menu por fases, terminal com o `trem` — rodando **na máquina do Google Colab** e aberto
**no seu navegador** por um link. Você clica e usa, sem instalar nada no seu computador.

> **Leia isto (engenharia honesta):** no Colab **não** emulamos o boot da ISO — QEMU sem
> aceleração (KVM) ficaria lento demais. Em vez disso montamos **o mesmo desktop** que vai
> na ISO, direto na VM do Colab, e mostramos por **noVNC + túnel Cloudflare**. O visual e o
> uso são os da distro; só não é o processo de boot. É um **demo** para sentir a cara do
> sistema — é um pouco lento (vídeo por VNC) e o Colab derruba sessões ociosas.

Para o uso OSINT de verdade no Colab (leve, rápido), use o notebook **otimizado**
`TOCAIA_OSINT_Linux.ipynb` (terminal + relatórios HTML inline).

**Passos:** menu *Ambiente de execução → Executar tudo* e, ao final, clique no link que aparecer.
Leva ~3–5 min na primeira montagem.


## 1 · Baixar a TOCAIA e gerar o tema (onça, menu, ícones)


In [ ]:
import os, subprocess
REPO = 'https://github.com/Ridd1kulusC0d3r/TOCAIALinux'
BRANCH = 'main'
if not os.path.exists('/content/TOCAIALinux'):
    subprocess.run(['git','clone','--depth','1','-b',BRANCH,REPO,'/content/TOCAIALinux'], check=True)
os.chdir('/content/TOCAIALinux')
# Pillow já vem no Colab; gera a marca (onça/face) e o tema direto na raiz '/'
subprocess.run(['python3','assets/gerar_marca.py','--destino','assets/gerado'], check=True)
subprocess.run(['python3','tema/gerar_tema.py','--destino','/'], check=True)
# empacota as ferramentas e instala o 'trem' no PATH
subprocess.run(['bash','scripts/empacotar.sh'], check=True)
subprocess.run('install -d /opt/tocaia/ferramentas', shell=True, check=True)
subprocess.run('cp catalogo/catalogo.json /opt/tocaia/ && cp ferramentas/*.pyz /opt/tocaia/ferramentas/ && cp -r casos wargames /opt/tocaia/ 2>/dev/null; true', shell=True)
subprocess.run('install -m0755 trem/trem.py /usr/local/bin/trem && ln -sf /usr/local/bin/trem /usr/local/bin/tocaia', shell=True, check=True)
print('TOCAIA preparada.')


## 2 · Instalar o desktop Xfce + VNC + o túnel
Baixa o Xfce enxuto, o servidor VNC, o noVNC e o cloudflared. (~2–3 min)


In [ ]:
%%bash
set -e
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq
# desktop enxuto + rasterizador de SVG (p/ o wallpaper) + VNC + noVNC
apt-get install -y -qq xfce4 xfce4-terminal xfce4-whiskermenu-plugin dbus-x11 \
   tigervnc-standalone-server tigervnc-common novnc websockify \
   librsvg2-bin fonts-dejavu-core >/dev/null 2>&1 || \
   apt-get install -y -qq xfce4 xfce4-terminal dbus-x11 tigervnc-standalone-server novnc websockify librsvg2-bin
# cloudflared (túnel https para abrir o desktop no navegador)
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
chmod +x /usr/local/bin/cloudflared
# rasteriza o wallpaper da onça (SVG -> PNG) para o Xfce usar
[ -f /usr/share/tocaia/wallpaper.svg ] && rsvg-convert -w 1600 -h 900 /usr/share/tocaia/wallpaper.svg -o /usr/share/tocaia/wallpaper.png || true
# aplica os padrões do Xfce (tema/menu/wallpaper) ao usuário atual
mkdir -p /root/.config && cp -r /etc/skel/.config/* /root/.config/ 2>/dev/null || true
echo 'desktop instalado.'


## 3 · Subir o desktop e pegar o link
Inicia o Xfce numa tela virtual, publica por noVNC e abre o túnel. **Clique no link** que aparecer (termina em `trycloudflare.com`).


In [ ]:
import subprocess, time, re, pathlib
# senha-vazia + sessão Xfce
pathlib.Path('/root/.vnc').mkdir(exist_ok=True)
pathlib.Path('/root/.vnc/xstartup').write_text('#!/bin/sh\nunset SESSION_MANAGER DBUS_SESSION_BUS_ADDRESS\nexec startxfce4\n')
subprocess.run('chmod +x /root/.vnc/xstartup', shell=True)
subprocess.run('vncserver -kill :1 >/dev/null 2>&1; true', shell=True)
subprocess.run('vncserver :1 -geometry 1600x900 -depth 24 -SecurityTypes None -localhost no',
               shell=True, check=True, env={**os.environ})
time.sleep(6)
# garante o wallpaper da onça em todos os monitores da sessão VNC
subprocess.run('''export DISPLAY=:1
ps=$(DISPLAY=:1 xfconf-query -c xfce4-desktop -l 2>/dev/null | grep last-image || true)
if [ -z "$ps" ]; then
  DISPLAY=:1 xfconf-query -c xfce4-desktop -p /backdrop/screen0/monitor0/workspace0/last-image -n -t string -s /usr/share/tocaia/wallpaper.png 2>/dev/null || true
else
  for p in $ps; do DISPLAY=:1 xfconf-query -c xfce4-desktop -p "$p" -s /usr/share/tocaia/wallpaper.png 2>/dev/null || true; done
fi
DISPLAY=:1 xfdesktop --reload 2>/dev/null || true''', shell=True)
# noVNC (porta 6080) apontando para o VNC (5901)
subprocess.Popen('websockify --web=/usr/share/novnc 6080 localhost:5901 >/content/novnc.log 2>&1', shell=True)
time.sleep(3)
# túnel Cloudflare
proc=subprocess.Popen('cloudflared tunnel --url http://localhost:6080 --no-autoupdate',
                      shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url=None
for _ in range(60):
    line=proc.stdout.readline()
    if not line: time.sleep(0.5); continue
    m=re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m: url=m.group(0); break
print('\n' + '='*60)
if url:
    print('✅ Abra o desktop da TOCAIA aqui (adicione /vnc.html):')
    print('   ' + url + '/vnc.html?autoconnect=1&resize=remote')
else:
    print('Não consegui o link do túnel; veja /content/novnc.log e rode esta célula de novo.')
print('='*60)


## Como usar
- Clique no link acima (com `/vnc.html?autoconnect=1&resize=remote`). Abre a **área de trabalho da TOCAIA**.
- No menu **TOCAIA** (canto superior), as ferramentas estão agrupadas pelas **6 fases**.
- Abra o **Terminal TOCAIA** e rode `trem`, `trem relogio --demo`, `trem placar ...`.

### Avisos
- É **demo**: vídeo por VNC é lento; não espere fluidez de desktop local.
- O Colab **encerra** sessões ociosas/longas — se cair, rode *Executar tudo* de novo.
- O link `trycloudflare.com` é **público e temporário**: não deixe dados sensíveis abertos na tela.

Para trabalhar de verdade: use o notebook **otimizado** (terminal + relatórios) ou a **ISO numa VM**.

---
**DEIVOSINT · OSINT no Jutsu** · Fonte pública · zero interação · base legal.
